# 🏆 Churn Prediction: Multi-Experiment Master Pipeline
Đây là bản Notebook hoàn chỉnh được tối ưu hóa cho bài toán Dự báo khách hàng rời mạng (Customer Churn). 

**Các điểm nổi bật:**
- **Anti-Leakage**: Loại bỏ các biến gây rò rỉ dữ liệu.
- **Multi-Model**: So sánh XGBoost và Logistic Regression.
- **Recall Optimization**: Tập trung vào việc không bỏ sót khách hàng rời mạng.
- **Explainability**: Tích hợp công cụ SHAP để giải thích mô hình.

In [ ]:
# 1. Install & Import Libraries
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn -q

import opendatasets as od
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import json
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# 2. Configuration & Dataset Download
dataset_url = "https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset"

if not os.path.exists("customer-churn-dataset"):
    # Try to load credentials from .secret if running on local/Colab
    if os.path.exists(".secret"):
        from dotenv import load_dotenv
        load_dotenv(".secret")
    
    username = os.getenv("KAGGLE_USERNAME")
    key = os.getenv("KAGGLE_KEY")
    
    if username and key:
        with open("kaggle.json", "w") as f:
            json.dump({"username": username, "key": key}, f)
        od.download(dataset_url)
        os.remove("kaggle.json")
    else:
        print("⚠️ Please provide Kaggle credentials to download the data.")

df = pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv")
print(f"✅ Loaded {len(df)} records.")

### 3. Training Logic (Multi-Experiment)

In [ ]:
def run_training_flow(df, model_type="xgboost"):
    # Clean names & Drop identifiers
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    # Drop Leakage Columns (last_interaction is a major leak in this dataset)
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    
    # Clean total_spend
    if 'total_spend' in df.columns:
        df['total_spend'] = pd.to_numeric(df['total_spend'], errors='coerce')
    df = df.dropna(subset=['churn', 'total_spend'])
    
    X = df.drop('churn', axis=1)
    y = df['churn']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Pipeline
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
    ])
    
    # Selection
    if model_type == "xgboost":
        cw = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
        clf = XGBClassifier(scale_pos_weight=cw, eval_metric='logloss')
        params = {'classifier__n_estimators': [100, 200], 'classifier__max_depth': [3, 5]}
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        params = {'classifier__C': [0.1, 1.0]}
    
    mlflow.set_experiment("Churn_Notebook_Comparison")
    with mlflow.start_run(run_name=f"NB_{model_type.upper()}"):
        search = RandomizedSearchCV(Pipeline([('pre', preprocessor), ('classifier', clf)]), params, n_iter=3, cv=3, scoring='recall')
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        
        # SHAP Explainability
        sample = X_test.sample(50)
        trans_df = pd.DataFrame(best_model.named_steps['pre'].transform(sample), columns=best_model.named_steps['pre'].get_feature_names_out())
        explainer = shap.Explainer(best_model.named_steps['classifier'])
        shap_values = explainer(trans_df)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, trans_df, show=False)
        plt.title(f"SHAP: {model_type}")
        plt.show()
        
        print(f"✅ {model_type.upper()} Finished. Recall: {recall_score(y_test, best_model.predict(X_test)):.4f}")

In [ ]:
for model in ["xgboost", "logistic_regression"]:
    run_training_flow(df, model)